In [1]:
import pyterrier as pt
from pathlib import Path
import pandas as pd
from pyterrier.measures import RR, nDCG, MAP, Recall
from fast_forward.encoder import TASBEncoder
import torch
from fast_forward.util import Indexer
from fast_forward.util.pyterrier import FFScore
from fast_forward.util.pyterrier import FFInterpolate
import numpy as np
print(pt.__version__)

0.12.1


Our experiment consists of four phases:
1) We only evaluate BM25 performance on the datasets (evSciFact, FiQA, TREC-COVID, NFCorpus)
2) We retrieve top-k (test several k values) and then use the pretrained neural reranker and evaluate the scores. Since the neural reranker is trained on MSMACRO we can't use it for evaluation.
3) We now check out the hybrid pipeline (interpolate BM25 with neural reranker) , so here we also test several k values and some α values
4) We need to test if the α value works across datasets (fix k value - found a good one in previous step) and tune α on one dataset (e.g Fiqa) and test it on the others.

## Create sparse indexes for BM25 for all datasets
We measure the performance of BM25 on our datasets

In [2]:
# first we import all our datasets:
# finance domain with some technical language usually for QA retrieval
fiqa = pt.get_dataset("irds:beir/fiqa")

# this dataset has structured and formal language (domain:science) and used for evidence retrieval
scifact = pt.get_dataset("irds:beir/scifact")

# arguana is a semantic dataset in the domain of debates/argumentation
arguana = pt.get_dataset("irds:beir/arguana")

# general QA dataset
nfcorpus = pt.get_dataset("irds:beir/nfcorpus")


In [3]:

fiqa_indexer = pt.IterDictIndexer(
    str(Path.cwd()),  # this will be ignored
    type=pt.index.IndexingType.MEMORY,
)
fiqa_index_ref = fiqa_indexer.index(fiqa.get_corpus_iter(), fields=["text"])


Java started (triggered by TerrierIndexer.__init__) and loaded: pyterrier.java, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]
beir/fiqa documents: 100%|██████████| 57638/57638 [00:13<00:00, 4126.64it/s]


In [4]:
# index for scifact
scifact_indexer = pt.IterDictIndexer(
    str(Path.cwd()),  # this will be ignored
    type=pt.index.IndexingType.MEMORY,
)
scifact_index_ref = scifact_indexer.index(scifact.get_corpus_iter(), fields=["text"])

beir/scifact documents: 100%|██████████| 5183/5183 [00:04<00:00, 1254.42it/s]


In [5]:
# index for nfcorpus
nfcorpus_indexer = pt.IterDictIndexer(
    str(Path.cwd()),  # this will be ignored
    type=pt.index.IndexingType.MEMORY,
)
nfcorpus_index_ref = nfcorpus_indexer.index(nfcorpus.get_corpus_iter(),fields=["text"])

beir/nfcorpus documents: 100%|██████████| 3633/3633 [00:02<00:00, 1340.35it/s]


In [6]:
# index for arguana
arguana_indexer = pt.IterDictIndexer(
    str(Path.cwd()),  # this will be ignored
    type=pt.index.IndexingType.MEMORY,
    meta={"docno": 100} # it has large ids
)
arguana_index_ref = arguana_indexer.index(arguana.get_corpus_iter(),fields=["text"])

beir/arguana documents: 100%|██████████| 8674/8674 [00:04<00:00, 1924.97it/s]


## Create the BM25 retrievers

In [7]:
# Phase 1 - evaluate retrieval only with BM25
# fiqa dataset
bm25_fiqa = pt.terrier.Retriever(fiqa_index_ref, wmodel="BM25")
testset_fiqa = pt.get_dataset("irds:beir/fiqa/test")
# pt.Experiment(
#     [bm25_fiqa],
#     testset_fiqa.get_topics(),
#     testset_fiqa.get_qrels(),
#     eval_metrics=[RR @ 10, nDCG @ 10, MAP @ 100],
# )


In [8]:
# scifact dataset
bm25_scifact = pt.terrier.Retriever(scifact_index_ref, wmodel="BM25")
testset_scifact = pt.get_dataset("irds:beir/scifact/test")
# pt.Experiment(
#     [bm25_scifact],
#     testset_scifact.get_topics(),
#     testset_scifact.get_qrels(),
#     eval_metrics=[RR @ 10, nDCG @ 10, MAP @ 100],
# )


In [9]:
# arguana dataset
bm25_arguana = pt.terrier.Retriever(arguana_index_ref, wmodel="BM25")
testset_arguana = pt.get_dataset("irds:beir/arguana")
# pt.Experiment(
#     [bm25_arguana],
#     testset_arguana.get_topics(),
#     testset_arguana.get_qrels(),
#     eval_metrics=[RR @ 10, nDCG @ 10, MAP @ 100],
# )

In [10]:
# nfcorpus dataset
bm25_nfcorpus = pt.terrier.Retriever(nfcorpus_index_ref, wmodel="BM25")
testset_nfcorpus = pt.get_dataset("irds:beir/nfcorpus/test")
# pt.Experiment(
#     [bm25_nfcorpus],
#     testset_nfcorpus.get_topics("text"),
#     testset_nfcorpus.get_qrels(),
#     eval_metrics=[RR @ 10, nDCG @ 10, MAP @ 100],
# )

## Neural reranker

We are going to use a pretrained neural model to rerank the top-k candidates. We are going to test several candidate depths to see how the performance reacts. I will test various candidate depths here but not on all the datasets, probably 2 of them are enough to make the assumptions neccessary.


In [11]:
q_encoder = d_encoder = TASBEncoder(
    device="cuda:0" if torch.cuda.is_available() else "cpu"
)

### Create the dense indexes with TAS-B

In [12]:
from fast_forward.index import OnDiskIndex, Mode
# create the empty dense indexes on disk

def create_index_onDisk(data_name):
    index_name = "ffindex_" + data_name + "_tasb.h5" 
    
    ff_index_path  = Path.cwd() / "ir_experiments" / index_name
    ff_index_path.parent.mkdir(exist_ok=True, parents=True)
    
    try:
        ff_index = OnDiskIndex(
            ff_index_path,
            query_encoder=q_encoder,
            mode=Mode.MAXP,
            max_id_length=64
        )
        return ff_index
    except Exception as e:
        return  OnDiskIndex.load(ff_index_path)
    
scifact_index=create_index_onDisk("scifact")
fiqa_index=create_index_onDisk("fiqa")
arguana_index=create_index_onDisk("arguana")
nfcorpus_index=create_index_onDisk("nfcorpus")

100%|██████████| 6161/6161 [00:00<00:00, 3242296.98it/s]


In [13]:
def docs_iter(dataset):
    for d in dataset.get_corpus_iter():
        yield {"doc_id": d["docno"], "text": d["text"]}


### **Don't run the following 4 lines - we have already generated the dense indexes**
(offline step)

In [ ]:
#fill in scifact dense index  !!!!!! DON'T RUN THIS I HAVE CREATED THIS ALREADY !!!!!!!!!
Indexer(scifact_index, d_encoder, batch_size=8).from_dicts(docs_iter(scifact))

In [ ]:
#fill in fiqa dense index !!!!!! DON'T RUN THIS I HAVE CREATED THIS ALREADY !!!!!!!!!
Indexer(fiqa_index, d_encoder, batch_size=8).from_dicts(docs_iter(fiqa))

In [ ]:
#fill in arguana dense index !!!!!! DON'T RUN THIS I HAVE CREATED THIS ALREADY !!!!!!!!!
Indexer(arguana_index, d_encoder, batch_size=8).from_dicts(docs_iter(arguana))

In [ ]:
# fill in nfcorpus dense index !!!! DON'T RUN THIS I HAVE CREATED THIS ALREADY !!!!
Indexer(nfcorpus_index, d_encoder, batch_size=8).from_dicts(docs_iter(nfcorpus))

### Load the indexes

In [14]:
# Load scifact index into memory
def LoadFromDisk(data_name):
    index_name = "ffindex_" + data_name + "_tasb.h5"
    index = OnDiskIndex.load(
    Path.cwd() / "ir_experiments" / index_name,
    query_encoder=q_encoder,
    mode=Mode.MAXP,
    )
    return index

    


In [15]:
# load indexes into memory
scifact_index = LoadFromDisk("scifact")
scifact_index = scifact_index.to_memory()

fiqa_index = LoadFromDisk("fiqa")
fiqa_index = fiqa_index.to_memory()

arguana_index =LoadFromDisk("arguana")
arguana_index = arguana_index.to_memory()

nfcorpus_index = LoadFromDisk("nfcorpus")
nfcorpus_index = nfcorpus_index.to_memory()

100%|██████████| 6161/6161 [00:00<00:00, 2368136.63it/s]


## Reranking BM25 results with TAS-B - Research Question 1
**Answer RQ1:Does adding neural reranking with TAS-B improve over plain BM25?**
To answer this question we will have a clean two-step comparision
1. BM25 Baseline
2. BM25 &rarr; TAS-B rerank

The experiment design is as follows :
1. We retrieve with BM25 (the top-k candidate documents)
2. We rerank the same candidates with TAS-B

Our metrics :
nDCG @ 10 and MRR @ 10 are our main metrics 
MAP @ 100, Recall @ 100 are secondary metrics

For answering **RQ1** we are going to keep a fixed candidate depth of k=100 which is pretty standard

In [ ]:
def rerank_only_experiment(dense_index, test_dataset, bm25, candidate_depths = [100]):
    # this function uses TAS-B as a pure reranker 
    # without interpolation - answers research question more cleanly
    ff_score = FFScore(dense_index)
    pipelines = [bm25]
    names = ["BM25"]

    for k in candidate_depths:
        pipe = (bm25 % k) >> ff_score
        pipelines.append(pipe)
        names.append(f"BM25 -> TAS-B rerank (k={k})")

    results = pt.Experiment(
        pipelines,
        test_dataset.get_topics("text"),
        test_dataset.get_qrels(),
        eval_metrics=[RR @ 10, nDCG @ 10, MAP @ 100, Recall @ 100],   # better than MAP@100 to checkout the k variable
        names=names,
    )
    return results

### Interpretation for RQ1 - SciFact Dataset
**write down interpretation**

In [ ]:
rerank_only_experiment(scifact_index, testset_scifact,bm25_scifact)

### Interpretation for RQ1 - FIQA Dataset
**write down interpretation**

In [ ]:
rerank_only_experiment(fiqa_index, testset_fiqa, bm25_fiqa)

### Interpretation for RQ1 - NFCorpus Dataset
**write down interpretation**

In [ ]:
rerank_only_experiment(nfcorpus_index,testset_nfcorpus,bm25_nfcorpus)

### Interpretation for RQ1 - ArguAna Dataset
**write down interpretation**

In [ ]:
rerank_only_experiment(arguana_index,testset_arguana,bm25_arguana)

---

# Research Question 2


## RQ2, Step 1: Tuning a global Alpha value

We tune a single global interpolation parameter α on a pooled set of development data from FiQA, NFCorpus, and SciFact ("train split"). For each candidate α, we compute the average nDCG@10 across these datasets and select the value that maximizes this average.

Later, we will measure the robustness of the value on different datasets and on subsets form the same datasets, that were not used in tuning the alpha value ("test split")

In [18]:
fixed_k = 100
alphas = np.round(np.arange(0.0, 1.01, 0.01), 2)

In [19]:
def findGlobalAlpha(alphas, bm25_list, ff_score_list, datasets, candidate_depth=100):
    best_alpha = None
    best_avg = -1

    for alpha in alphas:
        scores = []

        for bm25, ff_score, dataset in zip(bm25_list, ff_score_list, datasets):
            ff_int = FFInterpolate(alpha=alpha)
            pipe = (bm25 % candidate_depth) >> ff_score >> ff_int

            res = pt.Experiment(
                [pipe],
                dataset.get_topics("text"),
                dataset.get_qrels(),
                eval_metrics=["ndcg_cut_10"],
                names=[f"alpha={alpha}"]
            )

            scores.append(res.loc[0, "ndcg_cut_10"])

        avg_score = sum(scores) / len(scores)

        if avg_score > best_avg:
            best_avg = avg_score
            best_alpha = alpha

    return best_alpha

In [20]:
fiqa_tune_alpha = pt.get_dataset("irds:beir/fiqa/dev")
scifact_tune_alpha = pt.get_dataset("irds:beir/scifact/train")
nfcorpus_tune_alpha = pt.get_dataset("irds:beir/nfcorpus/dev")

In [21]:
tune_datasets = [fiqa_tune_alpha,scifact_tune_alpha,nfcorpus_tune_alpha]
bm25_per_dataset = [bm25_fiqa,bm25_scifact,bm25_nfcorpus]
ff_score_list = [FFScore(fiqa_index), FFScore(scifact_index), FFScore(nfcorpus_index)]

In [ ]:
best_global_alpha = findGlobalAlpha(alphas,bm25_per_dataset,ff_score_list,tune_datasets,fixed_k)

## RQ2, Step 2: Alpha performance on test split

In [16]:
alphas = np.round(np.arange(0.0, 1.01, 0.01), 2)  # 0.00, 0.05, ..., 1.00

datasets_config = [
    {
        "name": "FiQA",
        "bm25": bm25_fiqa,
        "ff_index": fiqa_index,
        "testset": testset_fiqa,
        "topic_variant": "text",
    },
    {
        "name": "SciFact",
        "bm25": bm25_scifact,
        "ff_index": scifact_index,
        "testset": testset_scifact,
        "topic_variant": "text",
    },
    {
        "name": "ArguAna",
        "bm25": bm25_arguana,
        "ff_index": arguana_index,
        "testset": testset_arguana,
        "topic_variant": "text",
    },
    {
        "name": "NFCorpus",
        "bm25": bm25_nfcorpus,
        "ff_index": nfcorpus_index,
        "testset": testset_nfcorpus,
        "topic_variant": "text",
    },
]

In [17]:
rows = []

for cfg in datasets_config:
    ff_score = FFScore(cfg["ff_index"])
    topics = cfg["testset"].get_topics(cfg["topic_variant"])
    qrels = cfg["testset"].get_qrels()

    # Retrieve + score once (these don't depend on alpha)
    scored = (cfg["bm25"] % fixed_k >> ff_score).transform(topics)

    for alpha in alphas:
        ff_int = FFInterpolate(alpha=alpha)
        interpolated = ff_int.transform(scored)

        # Evaluate using pt.Experiment on the already-interpolated frame
        result = pt.Experiment(
            [interpolated],
            topics,
            qrels,
            eval_metrics=[RR @ 10, nDCG @ 10, MAP @ 100],
            names=[f"{cfg['name']}_a{alpha}"],
        )

        rows.append({
            "dataset": cfg["name"],
            "alpha": alpha,
            "RR@10": result.iloc[0]["RR(rel=2)@10"] if "RR(rel=2)@10" in result.columns else result.iloc[0]["RR@10"],
            "nDCG@10": result.iloc[0]["nDCG@10"],
            "AP@100": result.iloc[0]["AP(rel=2)@100"] if "AP(rel=2)@100" in result.columns else result.iloc[0]["AP@100"],
        })

        print(f"[{cfg['name']}] alpha={alpha:.2f} → "
              f"RR@10={rows[-1]['RR@10']:.4f}  nDCG@10={rows[-1]['nDCG@10']:.4f}  AP@100={rows[-1]['AP@100']:.4f}")

# ── Save ───────────────────────────────────────────────────────
df_alpha = pd.DataFrame(rows)
df_alpha.to_csv("results_and_graphs/rq_2/alpha_sweep_results.csv", index=False)
print("\nSaved to results_and_graphs/rq_2/alpha_sweep_results.csv")
df_alpha

01:53:20.543 [main] WARN org.terrier.querying.ApplyTermPipeline -- The index has no termpipelines configuration, and no control configuration is found. Defaulting to global termpipelines configuration of 'Stopwords,PorterStemmer'. Set a termpipelines control to remove this warning.
[FiQA] alpha=0.00 → RR@10=0.3695  nDCG@10=0.3055  AP@100=0.2523
[FiQA] alpha=0.01 → RR@10=0.3707  nDCG@10=0.3065  AP@100=0.2531
[FiQA] alpha=0.02 → RR@10=0.3717  nDCG@10=0.3070  AP@100=0.2535
[FiQA] alpha=0.03 → RR@10=0.3737  nDCG@10=0.3082  AP@100=0.2551
[FiQA] alpha=0.04 → RR@10=0.3754  nDCG@10=0.3089  AP@100=0.2561
[FiQA] alpha=0.05 → RR@10=0.3767  nDCG@10=0.3101  AP@100=0.2575
[FiQA] alpha=0.06 → RR@10=0.3778  nDCG@10=0.3109  AP@100=0.2584
[FiQA] alpha=0.07 → RR@10=0.3803  nDCG@10=0.3118  AP@100=0.2597
[FiQA] alpha=0.08 → RR@10=0.3826  nDCG@10=0.3138  AP@100=0.2607
[FiQA] alpha=0.09 → RR@10=0.3803  nDCG@10=0.3133  AP@100=0.2603
[FiQA] alpha=0.10 → RR@10=0.3798  nDCG@10=0.3124  AP@100=0.2592
[FiQA] alpha=

,dataset,alpha,RR@10,nDCG@10,AP@100
0,FiQA,0.00,0.369461,0.305473,0.252348
1,FiQA,0.01,0.370710,0.306497,0.253109
2,FiQA,0.02,0.371681,0.306984,0.253525
3,FiQA,0.03,0.373727,0.308165,0.255066
4,FiQA,0.04,0.375404,0.308853,0.256113
...,...,...,...,...,...
399,NFCorpus,0.96,0.536765,0.324804,0.144002
400,NFCorpus,0.97,0.538725,0.324764,0.144120
401,NFCorpus,0.98,0.536295,0.324281,0.144008
402,NFCorpus,0.99,0.535148,0.322999,0.143842


## Experimenting with candidate depth in the reranking pipeline – Research Question 3
**How does candidate depth \(k\) affect reranking?**

We evaluate the effect of candidate depth by applying a BM25 → TAS-B reranking pipeline with varying values of \(k\) across multiple datasets. Increasing \(k\) allows the reranker to access a larger set of candidate documents, which may include additional relevant documents not present at smaller depths.

We evaluate performance using **nDCG@10** and **MRR@10** to measure the quality of the final ranked results. In addition, we report **Recall@k** to quantify the proportion of relevant documents retrieved within the candidate set at each depth, and **Recall@100** to assess how many relevant documents are ultimately promoted into the top-ranked results after reranking.

Recall@k is particularly important in this setting, as it reflects the coverage of relevant documents available for reranking and thus determines the upper bound of achievable performance. Recall@100 complements this by providing insight into how effectively the reranking model utilizes the additional candidates retrieved at larger depths.

By combining these metrics, we aim to identify the smallest candidate depth that achieves near-maximum ranking effectiveness while maintaining sufficient coverage of relevant documents.

In [ ]:
# this list will be used to find the best k for each of the datasets
candidate_depths = [100,200,300,400,500,800,1000]

In [ ]:
def recall_k_rerank_only(dense_index, test_dataset, bm25, candidate_depths):
    import pandas as pd

    ff_score = FFScore(dense_index)
    results_list = []

    for k in candidate_depths:
        pipeline = (bm25 % k) >> ff_score

        results = pt.Experiment(
            [pipeline],
            test_dataset.get_topics("text"),
            test_dataset.get_qrels(),
            eval_metrics=[Recall @ k],
            names=[f"BM25 -> TAS-B rerank (k={k})"]
        )

        # get recall value (only metric besides name)
        recall_col = [c for c in results.columns if c != "name"][0]
        recall_value = results.iloc[0][recall_col]

        results_list.append({
            "name": f"BM25 -> TAS-B rerank (k={k})",
            "Recall@k": recall_value
        })

    return pd.DataFrame(results_list)

### Results for SciFact - best-k in the two-stage pipeline
**interpret resutls**

In [ ]:
rerank_only_experiment(scifact_index, testset_scifact, bm25_scifact,candidate_depths)

In [ ]:
recall_k_rerank_only(scifact_index,testset_scifact,bm25_scifact,candidate_depths)

### Results for FiQA - best-k in the two-stage pipeline
**interpret resutls**

In [ ]:
rerank_only_experiment(fiqa_index, testset_fiqa,bm25_fiqa,candidate_depths)

In [ ]:
recall_k_rerank_only(fiqa_index, testset_fiqa, bm25_fiqa,candidate_depths)

### Results for NFCorpus - best-k in the two-stage pipeline
**interpret resutls**

In [ ]:
rerank_only_experiment(nfcorpus_index,testset_nfcorpus,"NFCorpus",bm25_nfcorpus,candidate_depths)

In [ ]:
recall_k_rerank_only(nfcorpus_index,testset_nfcorpus,bm25_nfcorpus,candidate_depths)

### Results for ArguAna - best-k in the two-stage pipeline
**interpret resutls**

In [ ]:
rerank_only_experiment(arguana_index,testset_arguana,bm25_arguana,candidate_depths)

In [ ]:
recall_k_rerank_only(arguana_index,testset_arguana,bm25_arguana,candidate_depths)

## Research question 3, Part B. Collect experiment metrics and save them in a csv_file for graphs

In [ ]:
import gc
import os

# I keep running out of memory, thus this abomination
def rerank_only_experiment_tofile(dense_index, test_dataset, bm25, candidate_depths, save_path):
    ff_score = FFScore(dense_index)

    topics = test_dataset.get_topics("text")
    qrels = test_dataset.get_qrels()
    metrics = [RR @ 10, nDCG @ 10, MAP @ 100, Recall @ 100]

    # Write header if file doesn't exist yet
    header_written = os.path.exists(save_path)

    # BM25 baseline first
    res = pt.Experiment([bm25], topics, qrels, eval_metrics=metrics, names=["BM25"])
    res.to_csv(save_path, index=False, mode='a' if header_written else 'w', header=not header_written)

    print("BM25 baseline done")
    del res; gc.collect()

    for k in candidate_depths:
        print(f"rerank_only_experiment for k={k}")
        pipe = (bm25 % k) >> ff_score
        res = pt.Experiment(
            [pipe], topics, qrels,
            eval_metrics=metrics,
            names=[f"BM25 -> TAS-B rerank (k={k})"]
        )
        res.to_csv(save_path, index=False, mode='a', header=False)
        print(f"  k={k} done")
        del res, pipe; gc.collect()


def recall_k_rerank_only_tofile(dense_index, test_dataset, bm25, candidate_depths, save_path):
    ff_score = FFScore(dense_index)
    topics = test_dataset.get_topics("text")
    qrels = test_dataset.get_qrels()

    header_written = os.path.exists(save_path)

    for k in candidate_depths:
        print(f"recall_k_rerank_only for k={k}")
        pipeline = (bm25 % k) >> ff_score

        results = pt.Experiment(
            [pipeline], topics, qrels,
            eval_metrics=[Recall @ k],
            names=[f"BM25 -> TAS-B rerank (k={k})"]
        )

        recall_col = [c for c in results.columns if c != "name"][0]
        recall_value = results.iloc[0][recall_col]

        row = pd.DataFrame([{"name": f"BM25 -> TAS-B rerank (k={k})", "Recall@k": recall_value}])
        row.to_csv(save_path, index=False, mode='a' if header_written else 'w', header=not header_written)
        header_written = True

        print(f"  k={k} done, Recall@k={recall_value:.4f}")

def rq3_collect_all_datasets(dataset_configs, candidate_depths, output_dir="results_and_graphs/rq_3/results"):
    out = Path(output_dir)
    out.mkdir(exist_ok=True, parents=True)

    for name, cfg in dataset_configs.items():
        print(f"\n{'='*40}\nRunning {name}...\n{'='*40}")

        rerank_path = out / f"{name}_rerank.csv"
        recall_path = out / f"{name}_recall.csv"

        recall_k_rerank_only_tofile(
            cfg["index"], cfg["testset"], cfg["bm25"],
            candidate_depths, save_path=rerank_path
        )
        gc.collect()

        recall_k_rerank_only_tofile(
            cfg["index"], cfg["testset"], cfg["bm25"],
            candidate_depths, save_path=recall_path
        )
        gc.collect()

        print(f"Saved {name} results to {out}")

In [ ]:
candidate_depths = list(range(50, 776, 25))

In [ ]:
dataset_configs = {
    "scifact":  {"index": scifact_index,  "testset": testset_scifact,  "bm25": bm25_scifact},
    "fiqa":     {"index": fiqa_index,     "testset": testset_fiqa,     "bm25": bm25_fiqa},
    "nfcorpus": {"index": nfcorpus_index, "testset": testset_nfcorpus, "bm25": bm25_nfcorpus},
    "arguana":  {"index": arguana_index,   "testset": testset_arguana,  "bm25": bm25_arguana},
}

rq3_collect_all_datasets(dataset_configs, candidate_depths)

---

# Old, useless code:
## Ignore interpretation and best k here, we don't need those results.
Reason is that we  need to interpret best-k in the reranking only pipeline **without the interpolation**

In [ ]:
def interpolationPipelineExperiment(dense_index,test_dataset,dataset_name,bm25,candidate_depths,alpha_value=0.5):
    # this one uses interpolation as well
    ff_score = FFScore(dense_index)
    ff_int = FFInterpolate(alpha=alpha_value) # we used a fixed alpha here maybe this needs to change
    pipelines = [bm25]
    names = ["BM25"]

    for k in candidate_depths:
        # generate the two stage pipe
        # bm25 retrieves top-k then tasb rescores them then we use interpolation to combine scores
        pipe = (bm25%k) >> ff_score >> ff_int
        pipelines.append(pipe)
        names.append(f"BM25 >> TAS-B >> interpolation")
    print(f"Results for {dataset_name} with alpha={alpha_value} ")
    results = pt.Experiment(
        pipelines,
        test_dataset.get_topics("text"),
        test_dataset.get_qrels(),
        eval_metrics=[RR @ 10, nDCG @ 10, MAP @ 100], # maybe here we need to remove MAP as a metric
        names=names,
    )
    print(results)

In [ ]:
# run experiment on any dataset I want
interpolationPipelineExperiment(scifact_index, testset_scifact, "SciFact", bm25_scifact,candidate_depths)

### Interpretation and best k - SciFact
A key result I can see from the neural reranking while adjusting the candidate depth for scifact is that performance saturates somewhere around 100-200 candidate depth (nDCG@10 and MAP@100 both go flat around 200). Increasing candidate depth beyond this depth doesn't yield any improvements. To balance efficiency and effectiveness , the **best k for SciFact is k=100**

In [ ]:
interpolationPipelineExperiment(fiqa_index, testset_fiqa, "FIQA", bm25_fiqa,candidate_depths)

### Interpretation and best k - FIQA
Although the strictly best k is k = 800 (nDCG@10 and MAP@100 highest values), going from k=1 to k=200 the gains are significant, however going from 200-1000 the gain is almost flat. Especially if I inspect the range from 500 to 1000 there is marginal gain. So taking into consideration the Efficiency vs Effectiveness tradeoff we believe that **k=200 , for FIQA** is the best for near-optimal performance

In [ ]:
# !!!!!!!!!!!!!!!!!!!!!!DON'T RUN THIS AGAIN IT TAKES FAR TOO LONG!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
interpolationPipelineExperiment(arguana_index,testset_arguana,"arguana",bm25_arguana,candidate_depths)

### Interpretation and best k - ArguAna
The results here are clear, after k=200 we see marginal if any improvement in nDCG@10 and MAP@100. It is worth noting that going deep in candidate depth here is very expensive and takes a lot of time due to the structure of the dataset. Taking this into consideration and the fact that performance saturates between k=100 and k=200 , we believe that the **best k for ArguAna is k=100**


In [ ]:
# !!!!!!!!!!!!!!!!!!!!!!DON'T RUN THIS AGAIN IT TAKES FAR TOO LONG!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
interpolationPipelineExperiment(nfcorpus_index,testset_nfcorpus,"NFCorpus",bm25_nfcorpus,candidate_depths)

### Interpretation and best k - NFCorpus
The results here are clear, performance improves substantially up to around k=100 and then it saturates with marginal improvement until k=200 (we see marginal if any improvement in nDCG@10 and MAP@100). Taking into consideration effectiveness vs efficiency, we believe that the **best k for NFCorpus is k=100** (approximately, as it might be near optimal)


## Phase 3 - Test robustness of alpha parameter
**my initial idea**
Tune alpha on one dataset, and apply it to all the others


Have a list for "best alpha" per dataset. Try this alpha on all other datasets. We used a fixed candidade depth in all the datasets. After the tests in phase 2, in order to balance efficiency with effectiveness we are going to use **fixed k value = 100** , which seems to be near optimal for all the datasets tested.

## Research Question 2 
We tune a single global interpolation parameter α on a pooled set of development data from FiQA, NFCorpus, and SciFact. For each candidate α, we compute the average nDCG@10 across these datasets and select the value that maximizes this average. This α is then applied unchanged to all test datasets, including ArguAna, to evaluate its robustness across domains. We are going to use **fixed k value = 100** , which seems to be near optimal for all the datasets tested.

In [ ]:
fixed_k_value = 100

The following function does exactly this.
1. tries different values of alpha &rarr; evaluates each alpha on the three datasets (dev or train parts of the dataset) and then averages the performance. It lastly picks the one with the best average performance.

In [ ]:
def findGlobalAlpha(alphas, bm25_list, ff_score_list, datasets, candidate_depth=100):
    #we 
    best_alpha = None
    best_avg = -1

    for alpha in alphas:
        scores = []

        for bm25, ff_score, dataset in zip(bm25_list, ff_score_list, datasets):
            ff_int = FFInterpolate(alpha=alpha)
            pipe = (bm25 % candidate_depth) >> ff_score >> ff_int

            res = pt.Experiment(
                [pipe],
                dataset.get_topics("text"),
                dataset.get_qrels(),
                eval_metrics=["ndcg_cut_10"],
                names=[f"alpha={alpha}"]
            )

            scores.append(res.loc[0, "ndcg_cut_10"])

        avg_score = sum(scores) / len(scores)

        if avg_score > best_avg:
            best_avg = avg_score
            best_alpha = alpha

    return best_alpha

### The process :
1. we use the dev or train dataset split of each dataset
2. we find the best global alpha among these splits.
3. we evaluate the performance of this alpha on the test splits of the respective datasets, and also include ArguAna (which wasn't used on the initial tuning as it only has a single split. - we just use it to evaluate)

In [ ]:
fiqa_tune_alpha = pt.get_dataset("irds:beir/fiqa/dev")
scifact_tune_alpha = pt.get_dataset("irds:beir/scifact/train")
nfcorpus_tune_alpha = pt.get_dataset("irds:beir/nfcorpus/dev")

test_alphas = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
tune_datasets = [fiqa_tune_alpha,scifact_tune_alpha,nfcorpus_tune_alpha]
bm25_per_dataset = [bm25_fiqa,bm25_scifact,bm25_nfcorpus]
ff_score_list = [FFScore(fiqa_index), FFScore(scifact_index), FFScore(nfcorpus_index)]


In [ ]:
# find the best globally tuned alpha value
best_global_alpha = findGlobalAlpha(test_alphas,bm25_per_dataset,ff_score_list,tune_datasets,fixed_k_value)

In [ ]:
fiqa_tune_alpha = pt.get_dataset("irds:beir/fiqa/dev")
scifact_tune_alpha = pt.get_dataset("irds:beir/scifact/train")
nfcorpus_tune_alpha = pt.get_dataset("irds:beir/nfcorpus/dev")

test_alphas = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
tune_datasets = [fiqa_tune_alpha,scifact_tune_alpha,nfcorpus_tune_alpha]
bm25_per_dataset = [bm25_fiqa,bm25_scifact,bm25_nfcorpus]
ff_score_list = [FFScore(fiqa_index), FFScore(scifact_index), FFScore(nfcorpus_index)]


**interpret results here**

## Ignore the following part of the code until refined (some of the results we don't actually need)
The following is related to my initial idea of answering the research question related to the globally tuned alpha. It is similar but I guess a bit weaker because it involves finding the "best tuned alpha" on each domain with GridSearch and then testing how it performs on each of the other domains. I am yet unsure if this is valuable to answering the research question compared to the above, we will think about it.

In [ ]:
# we pass in the dataset to find the best alpha for the specific dataset (e.g SciFact) then we use this alpha
def findBestAlpha(bm25, ff_score,dataset, candidate_depth=100):
    alphas = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
    ff_int = FFInterpolate(alpha=0.5)
    gs = pt.GridSearch(
    (bm25 % candidate_depth) >> ff_score >> ff_int,
    {ff_int: {"alpha": alphas}},
    dataset.get_topics("text"),
    dataset.get_qrels(),
    "map",
    verbose=True,
    )
    best_alpha = ff_int.get_parameter("alpha")
    print("Best alpha is:",best_alpha)
    return best_alpha


In [ ]:
# since I want to test how well the alpha parameter generalizes I can find it upon performing the process on the dataset
# of each domain and then testing this alpha on the other three datasets.
dataset_names = ["FIQA","SciFact","ArguAna","NFCorpus"]
all_datasets = [testset_fiqa, testset_scifact, testset_arguana, testset_nfcorpus]
all_bm25 = [bm25_fiqa,bm25_scifact,bm25_arguana,bm25_nfcorpus]
all_dense_indexes = [fiqa_index,scifact_index,arguana_index,nfcorpus_index]
best_alpha_per_pipeline = []

for i in range(len(all_datasets)): # we find the optimal on each of the datasets and test it on the other (different domain) datasets
    best_alpha_per_pipeline.append(findBestAlpha(all_bm25[i],FFScore(all_dense_indexes[i]),all_datasets[i],fixed_k_value))

In [ ]:
# Grid search takes a long time so just in case something goes wrong and we lose progress:
substitute_grid_search_values = [0.2,0.3,0.5,0.4]

In [ ]:
# Now best_alpha_per_pipeline can be used on each of the other datasets

for i in range(len(best_alpha_per_pipeline)):
    current_optimal = best_alpha_per_pipeline[i]
    for j in range(len(best_alpha_per_pipeline)):
        if i == j:
            continue
    # else call function that tests the result with specific alpha.
        print(f"\nTesting optimal alpha of {dataset_names[i]} with dataset: {dataset_names[j]}\n")
        interpolationPipelineExperiment(all_dense_indexes[j],all_datasets[j],dataset_names[j],all_bm25[j],[fixed_k_value],current_optimal)
 # this also takes quite some time to get results, due to ArguAna   

### Note 
maybe to properly test if it generalizes well we have to also check how the optimal alphas work on the same dataset or something, at least for the ones that have the test,train,dev split

In [ ]:
# To see if this generalization of alpha is any good we need to tune alpha e.g on Scifact/dev 
# and test its performance on Scifact/test. Then I can see if for example the alpha tuned on Fiqa and used on Scifact above (different domain than scifact)
# is good CROSS DOMAIN (need to compare it with alpha resulting from same domain)

# from the datasets we have used in our experiment, the ones that come pre-split into sub-datasets are:
# FiQA, SciFact and NF-Corpus so we are going to check these ones
fiqa_tune_alpha = pt.get_dataset("irds:beir/fiqa/dev")
scifact_tune_alpha = pt.get_dataset("irds:beir/scifact/train")
nfcorpus_tune_alpha = pt.get_dataset("irds:beir/nfcorpus/dev")


dataset_names_2 = ["FIQA","SciFact","NFCorpus"]
all_datasets_2 = [testset_fiqa, testset_scifact,  testset_nfcorpus]
all_bm25_2 = [bm25_fiqa,bm25_scifact,bm25_nfcorpus]
all_dense_indexes_2 = [fiqa_index,scifact_index,nfcorpus_index]

same_domain_tune_datasets = [fiqa_tune_alpha,scifact_tune_alpha,nfcorpus_tune_alpha]
same_domain_alpha = []

In [ ]:
# Now lets see what alpha our gradient descent finds in the tuning datasets
for i in range(len(all_datasets_2)): # we find the optimal on each of the datasets and test it on the other (different domain) datasets
    same_domain_alpha.append(findBestAlpha(all_bm25_2[i],FFScore(all_dense_indexes_2[i]),same_domain_tune_datasets[i],fixed_k_value))
    
print(same_domain_alpha)


In [ ]:
# evaluate on the other half (test-set of each of the datasets)
for i in range(len(same_domain_alpha)):
    current_optimal = same_domain_alpha[i]
    print(f"\nTesting optimal alpha of {dataset_names_2[i]} with the other split in same dataset\n")
    interpolationPipelineExperiment(all_dense_indexes_2[i],all_datasets_2[i],dataset_names_2[i],all_bm25_2[i],[fixed_k_value],current_optimal)